# Notebook for testing hypotheses

## Load enriched case log

In [1]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, confusion_matrix, f1_score

In [ ]:
INPUT_FILE_NAME_BERLIN = 'cases_Berlin-with-topics-and-agreement-scores-3EP-lawDraft-passedBills.csv'
INPUT_FILE_NAME_BRANDENBURG = 'cases_Brandenburg-with-topics-and-agreement-scores-preprocessed-3EP-lawDraft-passedBills.csv'
INPUT_FILE_NAME_BAWUE = 'cases_Baden-Württemberg-with-topics-and-agreement-scores-preprocessed-3EP-lawDraft-passedBills.csv'


df_brandenburg = pd.read_csv(INPUT_FILE_NAME_BRANDENBURG)
df_berlin = pd.read_csv(INPUT_FILE_NAME_BERLIN)
df_baWue = pd.read_csv(INPUT_FILE_NAME_BAWUE)

def sanitize_column_names(df):
    # Replace spaces and special characters with underscores
    df.columns = df.columns.str.replace(r'[()]', '_', regex=True)
    return df

df_brandenburg = sanitize_column_names(df_brandenburg)
df_berlin = sanitize_column_names(df_berlin)
df_baWue = sanitize_column_names(df_baWue)

print('Read', len(df_brandenburg), 'rows from', INPUT_FILE_NAME_BRANDENBURG)
print('Read', len(df_berlin), 'rows from', INPUT_FILE_NAME_BERLIN)
print('Read', len(df_baWue), 'rows from', INPUT_FILE_NAME_BAWUE)

df_brandenburg = df_brandenburg.dropna(axis=1, how='all')  # drop all completely empty columns
df_berlin = df_berlin.dropna(axis=1, how='all')  # drop all completely empty columns
df_baWue = df_baWue.dropna(axis=1, how='all')  # drop all completely empty columns

Read 440 rows from cases_Brandenburg-with-topics-and-agreement-scores-preprocessed-3EP-lawDraft-passedBills.csv
Read 501 rows from cases_Berlin-with-topics-and-agreement-scores-3EP-lawDraft-passedBills.csv
Read 335 rows from cases_Baden-Württemberg-with-topics-and-agreement-scores-preprocessed-3EP-lawDraft-passedBills.csv


In [3]:
bawue_average_cycle_time = df_baWue["duration"].mean()
bawue_average_cycle_time_upper_bound = bawue_average_cycle_time + 0.1*bawue_average_cycle_time
bawue_average_cycle_time_lower_bound = bawue_average_cycle_time - 0.1*bawue_average_cycle_time
print("BaWue average cycle time with threshold 10%:", bawue_average_cycle_time_upper_bound)

outcome = lambda row: row['duration'] > bawue_average_cycle_time_upper_bound

DATA_TYPES = {
    'slower_than_bawue_avg_tolerance': object,
    'faster_than_bawue_avg_tolerance': object,
    'in_bawue_avg_tolerance': object,
}

BaWue average cycle time with threshold 10%: 99.78477611940299


## Evaluate an explanation against the enriched case log

### Set outcome and explanation

In [4]:
# For df_berlin
df_berlin['outcome'] = df_berlin.apply(outcome, axis=1)
y_true_berlin = df_berlin['outcome']
print('Outcome distribution for Berlin:')
print(y_true_berlin.value_counts())

# For df_brandenburg
df_brandenburg['outcome'] = df_brandenburg.apply(outcome, axis=1)
y_true_brandenburg = df_brandenburg['outcome']
print('Outcome distribution for Brandenburg:')
print(y_true_brandenburg.value_counts())

# For df_baWue
df_baWue['outcome'] = df_baWue.apply(outcome, axis=1)
y_true_baWue = df_baWue['outcome']
print('Outcome distribution for BaWue:')
print(y_true_baWue.value_counts())



Outcome distribution for Berlin:
outcome
False    264
True     237
Name: count, dtype: int64
Outcome distribution for Brandenburg:
outcome
True     272
False    168
Name: count, dtype: int64
Outcome distribution for BaWue:
outcome
False    252
True      83
Name: count, dtype: int64


### Get scores

In [7]:
def test(df, y_true, rule): 
    df['explanation'] = df.apply(rule, axis=1)

    y_pred = df['explanation']

    precision = precision_score(y_true, y_pred, pos_label=True, zero_division=1)

    recall = recall_score(y_true, y_pred, pos_label=True)

    f1 = f1_score(y_true, y_pred, pos_label=True)

    print('Precision on case log:', round(precision, 3), ', Recall on case log:', round(recall, 3), ', f1 on case log:', round(f1, 3))

    nof_true_neg, nof_false_pos, nof_false_neg, nof_true_pos = confusion_matrix(y_true, y_pred).ravel()

    #print("\nTEST RESULTS")
    print('True positives (case delayed and predicted correctly):', nof_true_pos, nof_true_pos/y_true.sum())
    print('False positives (case not delayed, but predicted as such):', nof_false_pos, nof_false_pos/(y_true == False).sum())
    print('True negatives (case not delayed and not predicted as such):', nof_true_neg, nof_true_neg/(y_true == False).sum())
    print('False negatives (case delayed, but not predicted as such):', nof_false_neg, nof_false_neg/y_true.sum())
    print("\n")

### Manually Derived Rules (from the induced rules)

In [20]:
### from me / with modifications

# this could show that many activites are a stronger driver in brandenbrug than in berlin for delays...
# motivated by induced rule with highest recall for Brandenburg when hiding all temporal attributes
rule_berlin_1 = lambda row: row["event_count"] >= 9
rule_brandenburg_1 = lambda row: row["event_count"] >= 9
rule_baWue_1 = lambda row: row["event_count"] >= 9

# this shows the same as above, but more specific
# motivated by induced rules with relatively high recalls for Berlin and Brandenburg when hiding temporal attributes
rule_berlin_2 = lambda row: row["Ausschussberatung.count"] >= 3.0
rule_brandenburg_2 = lambda row: row["Sitzung.count"] >= 3.0
rule_baWue_2 = lambda row: row["Beschlussempfehlung und Bericht.count"] >= 3.0

# this is what we expect to be a main driver in delay in berlin
# motivated by rule with highest recall for berlin when including delay and hiding gesetz... 
rule_berlin_3 = lambda row: row['I. Lesung:Ausschussberatung.delay'] >= 28.0
rule_brandenburg_3 = lambda row: row['Sitzung:1. Lesung.delay'] <= -28.0
rule_baWue_3 = lambda row: row['Beschlussempfehlung und Bericht:Erste Beratung.delay'] <= -28.0 

# shows that in Brandenburg this is also a good indicator for delay but then the idea is, 
# that in berlin not much happens in between, while in Brandenburg Sitzungen happen... right?
# motivated by Brandenburg when allowing delay and hiding gesetz
rule_berlin_4 = lambda row: row['II. Lesung:I. Lesung.delay'] <= -60.0 
rule_brandenburg_4 = lambda row: row['2. Lesung:1. Lesung.delay'] <= -60.0 
rule_baWue_4 = lambda row: row['Zweite Beratung:Erste Beratung.delay'] <= -60.0 

# this motivates to do another comparison only with passed bills - but first do logistic regression quickly...
# even though the inducer did not get any rules with this, 
# we saw the big difference through the descriptive statistics 
# and obviously there could be many processes that just stop after the first reading for example and therefore do not take much time...
rule_berlin_5 = lambda row: row['case:is_passed_bill'] == 1
rule_brandenburg_5 = lambda row: row['case:is_passed_bill'] == 1
rule_baWue_5 = lambda row: row['case:is_passed_bill'] == 1

rules_berlin = [rule_berlin_1, rule_berlin_2, rule_berlin_3, rule_berlin_4, rule_berlin_5]
rules_brandenburg = [rule_brandenburg_1, rule_brandenburg_2, rule_brandenburg_3, rule_brandenburg_4, rule_brandenburg_5]
rules_baWue = [rule_baWue_1, rule_baWue_2, rule_baWue_3, rule_baWue_4, rule_baWue_5]


for i in range(len(rules_berlin)):
    print(f"-----------Testing rule set {i+1}--------------------")
    print("Berlin:")
    test(df_berlin, y_true_berlin, rules_berlin[i])
    print("Brandenburg:")
    test(df_brandenburg, y_true_brandenburg, rules_brandenburg[i])
    print("Baden-Württemberg:")
    test(df_baWue, y_true_baWue, rules_baWue[i])



-----------Testing rule set 1--------------------
Berlin:
Precision on case log: 0.619 , Recall on case log: 0.274 , f1 on case log: 0.38
True positives (case delayed and predicted correctly): 65 0.2742616033755274
False positives (case not delayed, but predicted as such): 40 0.15151515151515152
True negatives (case not delayed and not predicted as such): 224 0.8484848484848485
False negatives (case delayed, but not predicted as such): 172 0.7257383966244726


Brandenburg:
Precision on case log: 0.759 , Recall on case log: 0.662 , f1 on case log: 0.707
True positives (case delayed and predicted correctly): 180 0.6617647058823529
False positives (case not delayed, but predicted as such): 57 0.3392857142857143
True negatives (case not delayed and not predicted as such): 111 0.6607142857142857
False negatives (case delayed, but not predicted as such): 92 0.3382352941176471


Baden-Württemberg:
Precision on case log: 0.31 , Recall on case log: 0.325 , f1 on case log: 0.318
True positives (

KeyError: 'case:is_passed_bill'

In [ ]:
rule_berlin_2 = lambda row: row["case:staff"] <= 92478
rule_brandenburg_2 = lambda row: row["case:staff"] <= 80399
rule_baWue_2 = lambda row: row["case:staff"] <= 108111

rule_berlin_3 = lambda row: row['case:plenDays'] <= 15.93
rule_brandenburg_3 = lambda row: row['case:plenDays'] <= 19.53
rule_baWue_3 = lambda row: row['case:plenDays'] <= 24.18

rule_berlin_4 = lambda row: row['II. Lesung:I. Lesung.delay'] <= -30.0 
rule_brandenburg_4 = lambda row: row['2. Lesung:1. Lesung.delay'] <= -30.0 
rule_baWue_4 = lambda row: row['Zweite Beratung:Erste Beratung.delay'] <= -30.0 

rule_berlin_5 = lambda row: row['case:is_passed_bill'] == 1
rule_brandenburg_5 = lambda row: row['case:is_passed_bill'] == 1
rule_baWue_5 = lambda row: row['case:is_passed_bill'] == 1

rules_berlin = [rule_berlin_2, rule_berlin_3, rule_berlin_4, rule_berlin_5]
rules_brandenburg = [rule_brandenburg_2, rule_brandenburg_3, rule_brandenburg_4, rule_brandenburg_5]
rules_baWue = [rule_baWue_2, rule_baWue_3, rule_baWue_4, rule_baWue_5]

for i in range(len(rules_berlin)):
    print(f"-----------Testing rule set {i+1}--------------------")
    test(df_berlin, y_true_berlin, rules_berlin[i])
    test(df_brandenburg, y_true_brandenburg, rules_brandenburg[i])
    test(df_baWue, y_true_baWue, rules_baWue[i])




-----------Testing rule set 1--------------------


KeyError: 'case:squire'